In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Media-Channel-Retention-V1 fixed diagnostic

Run all once. It reads exactly the retained Window-State-MSE-V1 evaluation artifacts, performs 20 new VAE encodes, and records all fixed success/failure slots. It does not generate, decode a latent video, create/decode/re-encode an MP4, add an attack, calibrate, tune, or run a deployment receiver. Reading MP4 bytes is limited to SHA-256 binding. Process completion is not a method PASS.

In [ ]:
from pathlib import Path
import datetime, json, sys
SOURCE_SHA = '1e476f8c3d51ed0f646343ac261fe02127c3368c'
INPUT_ROOT = Path('/content/drive/MyDrive/Video-WM/Window-State-MSE-V1/window_state_mse_v1_20260922T174349464768Z')
OUTPUT_PARENT = Path('/content/drive/MyDrive/Video-WM/Media-Channel-Retention-V1')
OUTPUT_PARENT.mkdir(parents=True, exist_ok=True)
stamp = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUTPUT = OUTPUT_PARENT / ('media_channel_retention_v1_' + stamp)
OUTPUT.mkdir(exist_ok=False)
(OUTPUT / 'setup_receipt.json').write_text(json.dumps(dict(source_commit=SOURCE_SHA, input_root=str(INPUT_ROOT), python=sys.version, executable=sys.executable, status='SETUP_STARTED'), indent=2))
print('fixed input:', INPUT_ROOT, flush=True)
print('same-run output:', OUTPUT, flush=True)


In [ ]:
import subprocess, sys, json
SETUP_LOG = OUTPUT / 'setup.log'
def logged_run(command, check=True):
    with SETUP_LOG.open('a', encoding='utf-8') as log:
        line = 'COMMAND ' + repr(command) + '\n'
        print(line, end='', flush=True); log.write(line); log.flush()
        child = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in child.stdout:
            print(line, end='', flush=True); log.write(line); log.flush()
        returncode = child.wait()
        log.write('EXIT ' + str(returncode) + '\n'); log.flush()
    if check and returncode:
        (OUTPUT / 'setup_failure.json').write_text(json.dumps(dict(command=command, returncode=returncode), indent=2))
        raise subprocess.CalledProcessError(returncode, command)
    return subprocess.CompletedProcess(command, returncode)
print('Python:', sys.version, 'Executable:', sys.executable, flush=True)
import importlib.metadata, subprocess, sys
print('Python:', sys.version, flush=True)
print('Executable:', sys.executable, flush=True)
logged_run([sys.executable, '-m', 'pip', '--version'], check=True)
logged_run(['apt-get', 'update', '-qq'], check=True)
logged_run(['apt-get', 'install', '-y', '-qq', 'ffmpeg'], check=True)
def version(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None
print('torch before install:', version('torch'), flush=True)
if version('torch') != '2.11.0+cu128':
    logged_run([sys.executable, '-m', 'pip', 'install', 'torch==2.11.0', 'torchvision', '--index-url', 'https://download.pytorch.org/whl/cu128'], check=True)
logged_run([sys.executable, '-m', 'pip', 'install', 'diffusers==0.40.0', 'transformers', 'accelerate', 'ftfy', 'sentencepiece', 'safetensors', 'huggingface_hub', 'numpy', 'Pillow'], check=True)
check_code = """
import importlib.metadata, sys, torch, diffusers
print('Fresh process Python:', sys.version, flush=True)
print('Fresh process executable:', sys.executable, flush=True)
for name in ('torch', 'torchvision', 'diffusers', 'transformers', 'accelerate', 'ftfy', 'sentencepiece', 'safetensors', 'huggingface_hub', 'numpy', 'Pillow'):
    try:
        value = importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        value = None
    print(name + ':', value, flush=True)
assert str(torch.__version__) == '2.11.0+cu128', torch.__version__
assert diffusers.__version__ == '0.40.0', diffusers.__version__
"""
check_code = check_code.replace("assert str(torch.__version__)", "from pathlib import Path\nimport importlib.metadata, json, sys\ninfo = dict(python=sys.version, executable=sys.executable, packages={})\nfor package in ('torch', 'torchvision', 'diffusers', 'transformers', 'accelerate', 'ftfy', 'sentencepiece', 'safetensors', 'huggingface_hub', 'numpy', 'Pillow'):\n    try: info['packages'][package] = importlib.metadata.version(package)\n    except importlib.metadata.PackageNotFoundError: info['packages'][package] = None\nPath(RECEIPT_PATH).write_text(json.dumps(info, indent=2))\n".replace('RECEIPT_PATH', repr(str(OUTPUT / 'environment_setup.json'))) + "assert str(torch.__version__)")
logged_run([sys.executable, '-u', '-c', check_code], check=True)


In [ ]:
import subprocess
REPO = Path('/content/SC-SSTW-Media-Channel-Retention-' + stamp)
logged_run(['git', 'clone', '--filter=blob:none', 'https://github.com/RICHAAARC/SC-SSTW.git', str(REPO)])
logged_run(['git', '-C', str(REPO), 'checkout', '--detach', SOURCE_SHA])
actual = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
assert actual == SOURCE_SHA
print('source commit:', actual, flush=True)


In [ ]:
import subprocess, sys
check_code = """
import torch
print('torch:', torch.__version__, 'cuda:', torch.version.cuda, flush=True)
if not torch.cuda.is_available():
    raise RuntimeError('This fixed real Wan run requires a CUDA runtime')
print('device:', torch.cuda.get_device_name(0), flush=True)
"""
logged_run([sys.executable, "-u", "-c", check_code], check=True)


In [ ]:
import json, os, subprocess, sys
CONFIG = REPO / 'experiments/wan_state_clock/configs/media_channel_retention_v1.json'
env = os.environ.copy(); env['PYTHONUNBUFFERED'] = '1'
command = [sys.executable, '-u', '-m', 'experiments.wan_state_clock.media_channel_retention_run', '--config', str(CONFIG), '--output', str(OUTPUT)]
print('diagnostic argv:', command, flush=True)
print('diagnostic output:', OUTPUT, flush=True)
completed = subprocess.run(command, cwd=REPO, env=env, check=False)
print('diagnostic returncode:', completed.returncode, flush=True)
(OUTPUT / 'execution_receipt.json').write_text(json.dumps(dict(command=command, returncode=completed.returncode, result_path=str(OUTPUT / 'result.json')), indent=2))
if not (OUTPUT / 'result.json').exists():
    raise FileNotFoundError('runner produced no retained result.json')


In [ ]:
import json
result_path = OUTPUT / 'result.json'
result = json.loads(result_path.read_text())
print('status:', result.get('status'))
print('fixed input audit:', result.get('input_root_audit'))
print('call accounting:', result.get('call_accounting'))
print('slot accounting:', result.get('slot_accounting'))
for case_id, case in result.get('cases', {}).items():
    print('case:', case_id, 'status:', case.get('status'), 'subprocess:', case.get('subprocess'))
    print(' case calls:', case.get('call_accounting'), 'failures:', case.get('failures'))
    for arm, row in case.get('trajectories', {}).items():
        print(' ', arm, row.get('status'), {name: layer.get('status') for name, layer in row.get('layers', {}).items()})
        if row.get('failures'): print('   failures:', row['failures'])
print('marked-minus-OFF slots:', len(result.get('marked_off_increments', [])))
print('adjacent increment-change slots:', len(result.get('adjacent_increment_changes', [])))
print('terminal-to-RGB8 combined slots:', len(result.get('terminal_to_rgb8_combined_increment_changes', [])))
def delta(row, field):
    return row.get('comparison', {}).get('scalars', {}).get(field, {}).get('delta')
increments = {
    (row['case_id'], row['marked_arm'], row['partition'], row['layer']): row
    for row in result.get('marked_off_increments', [])
}
changes = {
    (row['case_id'], row['marked_arm'], row['partition'], row['before_layer'], row['after_layer']): row
    for row in result.get('adjacent_increment_changes', [])
}
layers = ('TERMINAL', 'FLOAT_RGB_REENCODE', 'RGB8_NO_CODEC_REENCODE', 'EXISTING_MP4_G0')
transitions = tuple(zip(layers, layers[1:]))
for case_id in ('eval_p2_s2', 'eval_p3_s3'):
    for arm in ('LEGACY_SINGLE46', 'MSE_SINGLE46', 'LEGACY_MULTI44_46', 'MSE_MULTI44_46'):
        for partition in ('TOTAL', 'A', 'B'):
            print('increment table:', case_id, arm, partition)
            for layer in layers:
                row = increments.get((case_id, arm, partition, layer), {})
                print(' ', layer, 'matched marked-OFF=', delta(row, 'matched_score'), 'innovation marked-OFF=', delta(row, 'state_innovation_mean'), 'status=', row.get('comparison', {}).get('status', 'UNDEFINED'))
            for before, after in transitions:
                row = changes.get((case_id, arm, partition, before, after), {})
                print(' ', before + ' -> ' + after, 'matched increment change=', delta(row, 'matched_score'), 'innovation increment change=', delta(row, 'state_innovation_mean'), 'status=', row.get('comparison', {}).get('status', 'UNDEFINED'))
print('All 11-window q values and their deltas are retained in result.json fields cases.*.trajectories.*.layers.*.measurements.*.q_by_window, marked_off_increments, and adjacent_increment_changes.')
print('top-level retained failures:', result.get('failures', []))
print('full result:', result_path)
